In [25]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions and help with tasks."),
    ("user", "{input}")
])

chain = prompt | llm

result = chain.invoke({"input": "What is Gen ai?"})
print(result)

content='"Gen AI" typically refers to "Generative AI," which is a subset of artificial intelligence focused on creating new content or data that resembles existing data. This can include generating text, images, music, and other forms of media. Generative AI models are trained on large datasets and learn to produce outputs that are coherent and contextually relevant.\n\nSome common applications of generative AI include:\n\n1. **Text Generation**: Models like OpenAI\'s GPT (Generative Pre-trained Transformer) can generate human-like text, write stories, answer questions, and assist in various writing tasks.\n\n2. **Image Generation**: Tools like DALL-E and Midjourney can create images from textual descriptions, allowing users to generate artwork or visual content based on their prompts.\n\n3. **Music Composition**: AI can compose original music pieces by learning from existing compositions and styles.\n\n4. **Video Generation**: Some advanced models can generate video content or manipul

In [26]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama3-8b-8192", temperature=0)

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions and help with tasks."),
    ("user", "{input}")
])

chain = prompt | llm    
chain.invoke({"input": "What is Gen ai?"})

AIMessage(content="Gen AI, also known as Generalized Artificial Intelligence, refers to a hypothetical future AI system that possesses the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence.\n\nIn other words, Gen AI would be a highly advanced AI system that can:\n\n1. Reason and learn from vast amounts of data, just like humans do.\n2. Apply this knowledge to solve complex problems, make decisions, and take actions.\n3. Adapt to new situations and learn from experience.\n4. Communicate effectively with humans, understanding and generating natural language.\n5. Demonstrate common sense, creativity, and intuition.\n\nGen AI would be a significant departure from current AI systems, which are typically designed to perform specific tasks, such as:\n\n* Narrow AI (e.g., image recognition, language translation, or playing chess).\n* Weak AI (e.g., virtual assistants, like Siri or Alexa).\n\nThe development of Gen AI is still in its i

In [27]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_core.documents import Document


docs = [
    Document(page_content="Hello, world!", metadata={"source": "test"}),
    Document(page_content="Hello, world!", metadata={"source": "test"}),
]
embeddings.embed_query("Hello, world!s")


#embeddings.embed_documents(docs)

[0.0044154515489935875,
 -0.007479158695787191,
 0.030748477205634117,
 0.004905644804239273,
 -0.0383836068212986,
 -0.02456907369196415,
 -0.014104193076491356,
 0.04658320173621178,
 -0.018463941290974617,
 -0.038353897631168365,
 0.0011967024765908718,
 -0.02497014030814171,
 -0.012447934597730637,
 -0.01178691629320383,
 0.014178464189171791,
 0.028000425547361374,
 -0.05920938774943352,
 0.03449176996946335,
 0.01070254947990179,
 0.006569330580532551,
 0.05199018120765686,
 0.017483554780483246,
 -0.0073751783929765224,
 0.026425866410136223,
 0.04729620739817619,
 -0.00924682430922985,
 -0.025341499596834183,
 0.037522055208683014,
 0.014386425726115704,
 -0.0399581678211689,
 0.034402646124362946,
 -0.05294085666537285,
 -0.009551338851451874,
 -0.006379937753081322,
 -0.0104054631665349,
 0.030080031603574753,
 0.004066374618560076,
 -0.0032029664143919945,
 -0.00657304422929883,
 -0.030421681702136993,
 -0.01250735204666853,
 -0.004950207658112049,
 0.018968988209962845,
 0.

In [28]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

index = faiss.IndexFlatL2(1536)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_texts(["Trying texttosqlagent","Trying sqlagent"])





['47bb20ce-d972-4518-8fa0-c7886da077e9',
 '95b39264-40e9-4ae0-a284-1bb162b2d377']

In [ ]:
from langchain_community.document_loaders import TextLoader
filepath="TEXT.txt"
loader=TextLoader(filepath)
print(len(loader.load()))
pages=loader.load()



1


In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)

split_docs = splitter.split_documents(pages)
len(split_docs)

index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

#vector_store.add_documents(documents=split_docs)

In [31]:
retriever=vector_store.as_retriever(
    search_kwargs={"k": 10} #hyperparameter
)

In [32]:
from dotenv import load_dotenv
import os

load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-1.5-flash')

In [34]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

import pprint
pprint.pprint(prompt.messages)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
#context(retriever),prompt(hub),model(google),parser(langchain)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

rag_chain.invoke("what is GENAIs?")

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


AssertionError: 